In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1) Quantum core: 2-qubit state, Pauli analyzers, Born sampling
# ============================================================

# Pauli matrices
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

def bell_singlet():
    """
    |psi-> = (|01> - |10>)/sqrt(2) in basis |00>,|01>,|10>,|11>.
    """
    psi = np.zeros(4, dtype=complex)
    psi[1] = 1/np.sqrt(2)
    psi[2] = -1/np.sqrt(2)
    return psi

def observable(angle):
    """
    Analyzer observable with eigenvalues ±1.
    We use the polarization-like 'double-angle' map:
      O(angle) = cos(2a) σz + sin(2a) σx
    This gives E(a,b) = -cos(2(a-b)) for the singlet.
    """
    return np.cos(2*angle) * sz + np.sin(2*angle) * sx

def eig_projectors(O):
    """
    Return (+1, ket_plus, P_plus) and (-1, ket_minus, P_minus)
    for a Hermitian 2x2 observable.
    """
    w, V = np.linalg.eigh(O)   # eigenvalues w, eigenvectors columns of V
    # Identify +1 and -1 eigenvectors robustly
    idx_plus  = np.argmax(w)
    idx_minus = 1 - idx_plus
    v_plus  = V[:, idx_plus]
    v_minus = V[:, idx_minus]
    P_plus  = np.outer(v_plus,  v_plus.conj())
    P_minus = np.outer(v_minus, v_minus.conj())
    return (+1, v_plus, P_plus), (-1, v_minus, P_minus)

def joint_outcome_table(psi, a, b):
    """
    Build the 4 joint outcomes (sA,sB) and their Born probabilities:
      p(sA,sB) = <psi| (P^A_{sA}(a) ⊗ P^B_{sB}(b)) |psi>
    """
    (sAp, vAp, PAp), (sAm, vAm, PAm) = eig_projectors(observable(a))
    (sBp, vBp, PBp), (sBm, vBm, PBm) = eig_projectors(observable(b))

    outcomes = []
    probs = []
    for (sA, vA, PA) in [(sAp, vAp, PAp), (sAm, vAm, PAm)]:
        for (sB, vB, PB) in [(sBp, vBp, PBp), (sBm, vBm, PBm)]:
            P = np.kron(PA, PB)              # 4x4 projector onto joint outcome
            p = np.vdot(psi, P @ psi).real   # Born probability (real scalar)
            outcomes.append((sA, sB, vA, vB, P))
            probs.append(p)

    probs = np.array(probs, dtype=float)
    probs = probs / probs.sum()
    return outcomes, probs

def sample_joint_outcome(psi, a, b, rng=np.random.default_rng()):
    """
    One trial of a *joint* projective measurement on two qubits.

    Returns:
      sA, sB: ±1 outcomes
      vA, vB: the local eigenkets corresponding to those outcomes/settings
      psi_post: collapsed normalized joint state
      probs: the 4 joint probabilities for this (a,b)
    """
    outcomes, probs = joint_outcome_table(psi, a, b)
    k = rng.choice(len(outcomes), p=probs)   # sample outcome index
    sA, sB, vA, vB, P = outcomes[k]

    # Collapse (Lüders rule): psi_post ∝ P |psi>
    psi_post = P @ psi
    n = np.linalg.norm(psi_post)
    if n > 0:
        psi_post /= n

    return dict(
        sA=sA, sB=sB,
        vA=vA, vB=vB,
        psi_pre=psi.copy(),
        psi_post=psi_post,
        outcomes=outcomes,
        probs=probs
    )

def reduced_density_matrix(psi, which="A"):
    """
    Reduced density matrix for pure 2-qubit state psi (length 4).
    psi reshaped as 2x2: rows=A, cols=B.
    """
    Psi = psi.reshape(2, 2)
    if which.upper() == "A":
        return Psi @ Psi.conj().T          # Tr_B
    else:
        return Psi.conj().T @ Psi          # Tr_A

def bloch_vector(rho):
    """Return Bloch vector components (rx, ry, rz) for 2x2 density matrix."""
    rx = np.trace(rho @ sx).real
    ry = np.trace(rho @ sy).real
    rz = np.trace(rho @ sz).real
    return np.array([rx, ry, rz], dtype=float)

# ============================================================
# 2) Rendering layer: map local qubit ket [q+, q-] -> αβ -> abc
#    (educational embedding)
# ============================================================

def clarke_inv(v_alpha, v_beta):
    """
    Inverse Clarke (no zero-sequence):
      Va = v_alpha
      Vb = -1/2 v_alpha + sqrt(3)/2 v_beta
      Vc = -1/2 v_alpha - sqrt(3)/2 v_beta
    """
    va = v_alpha
    vb = -0.5*v_alpha + (np.sqrt(3)/2)*v_beta
    vc = -0.5*v_alpha - (np.sqrt(3)/2)*v_beta
    return va, vb, vc

def qubit_to_alphabeta(q, t, omega=1.0):
    """
    Embedding:
      v(t) = q_plus e^{+j ω t} + q_minus e^{-j ω t}
    where q = [q_plus, q_minus] is a normalized ket.
    Returns real (v_alpha(t), v_beta(t)).
    """
    q_plus, q_minus = q[0], q[1]
    v = q_plus*np.exp(1j*omega*t) + q_minus*np.exp(-1j*omega*t)
    return v.real, v.imag

def qubit_to_3phase(q, t, omega=1.0):
    v_alpha, v_beta = qubit_to_alphabeta(q, t, omega=omega)
    va, vb, vc = clarke_inv(v_alpha, v_beta)
    return (va, vb, vc), (v_alpha, v_beta)

def random_pure_qubit(rng=np.random.default_rng()):
    """
    Sample a random pure qubit state (good enough for visualization ensemble).
    """
    u = rng.uniform(0, 1)
    phi = rng.uniform(0, 2*np.pi)
    theta = np.arccos(1 - 2*u)  # produces sin(theta) distribution
    ket = np.array([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)], dtype=complex)
    return ket / np.linalg.norm(ket)

# ============================================================
# 3) Plot helpers
# ============================================================

def plot_joint_probs(ax, outcomes, probs, title="Joint probs"):
    # Map probs into a 2x2 in order (+,+),(+,-),(-,+),(-,-)
    p = {(sA, sB): pr for (sA, sB, *_), pr in zip(outcomes, probs)}
    grid = np.array([[p[(+1,+1)], p[(+1,-1)]],
                     [p[(-1,+1)], p[(-1,-1)]]])
    im = ax.imshow(grid, aspect='equal')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels([r"$s_B=+1$", r"$s_B=-1$"])
    ax.set_yticklabels([r"$s_A=+1$", r"$s_A=-1$"])
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{grid[i,j]:.3f}", ha="center", va="center",
                    color=("white" if grid[i,j] > 0.5 else "black"))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

def plot_equator_axes(ax, angles, labels, title="Analyzer axes (x–z equator, double-angle)"):
    # In x–z plane, measurement axis n(a) = (sin 2a, 0, cos 2a)
    th = np.linspace(0, 2*np.pi, 400)
    ax.plot(np.cos(th), np.sin(th), lw=1.0)
    for ang, lab in zip(angles, labels):
        x = np.sin(2*ang)
        z = np.cos(2*ang)
        ax.arrow(0, 0, x, z, head_width=0.05, head_length=0.07, length_includes_head=True)
        ax.text(1.10*x, 1.10*z, lab, ha="center", va="center", fontsize=9)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.15, 1.15)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("x"); ax.set_ylabel("z")
    ax.set_title(title)

def plot_alphabeta(ax, v_alpha, v_beta, title):
    ax.plot(v_alpha, v_beta, lw=1.0)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel(r"$v_\alpha$")
    ax.set_ylabel(r"$v_\beta$")
    ax.set_title(title)

def plot_abc(ax, t, va, vb, vc, title):
    ax.plot(t, va, lw=1.0, label="Va")
    ax.plot(t, vb, lw=1.0, label="Vb")
    ax.plot(t, vc, lw=1.0, label="Vc")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)

def plot_mixed_state_ensemble(ax_ab, ax_abc, t, omega, rng, n=40):
    """
    Visualize 'no unique local waveform' by sampling an ensemble of pure kets
    (since rho=I/2 is maximally mixed).
    """
    for _ in range(n):
        q = random_pure_qubit(rng)
        (va, vb, vc), (v_alpha, v_beta) = qubit_to_3phase(q, t, omega=omega)
        ax_ab.plot(v_alpha, v_beta, lw=0.6, alpha=0.15)
        ax_abc.plot(t, va, lw=0.5, alpha=0.08)
        ax_abc.plot(t, vb, lw=0.5, alpha=0.08)
        ax_abc.plot(t, vc, lw=0.5, alpha=0.08)

    ax_ab.set_aspect('equal', adjustable='box')
    ax_ab.grid(True, alpha=0.3)
    ax_ab.set_title("Before measurement: local state is mixed (ρ=I/2)\nEnsemble of possible αβ trajectories")
    ax_ab.set_xlabel(r"$v_\alpha$"); ax_ab.set_ylabel(r"$v_\beta$")

    ax_abc.grid(True, alpha=0.3)
    ax_abc.set_title("Before measurement: no unique 3-phase waveform\n(ensemble visualization)")
    ax_abc.set_xlabel("t")

# ============================================================
# 4) Demo: one setting pair, show before/after collapse & waveforms
# ============================================================

rng = np.random.default_rng(5)

psi = bell_singlet()

# Choose one (a,b) setting pair (standard CHSH angles)
a = 0.0
b = np.pi/8

# Compute joint table and sample one outcome
res = sample_joint_outcome(psi, a, b, rng=rng)

sA, sB = res["sA"], res["sB"]
vA_ket, vB_ket = res["vA"], res["vB"]

# Reduced states (before/after)
rhoA_pre  = reduced_density_matrix(res["psi_pre"],  "A")
rhoB_pre  = reduced_density_matrix(res["psi_pre"],  "B")
rhoA_post = reduced_density_matrix(res["psi_post"], "A")
rhoB_post = reduced_density_matrix(res["psi_post"], "B")

rA_pre, rB_pre   = bloch_vector(rhoA_pre),  bloch_vector(rhoB_pre)
rA_post, rB_post = bloch_vector(rhoA_post), bloch_vector(rhoB_post)

# Render time base
N = 1200
t = np.linspace(0, 2*np.pi, N, endpoint=False)
omega = 1.0

# After collapse: render the *realized* local eigenkets as abc and αβ
(alice_abc, alice_ab) = qubit_to_3phase(vA_ket, t, omega=omega)
(bob_abc,   bob_ab)   = qubit_to_3phase(vB_ket, t, omega=omega)

# ---- Figure: analyzers, joint probs, Bloch pre/post
fig1, axs = plt.subplots(1, 3, figsize=(14, 4))

plot_equator_axes(
    axs[0],
    angles=[a, b],
    labels=[f"A: a={np.degrees(a):.1f}°", f"B: b={np.degrees(b):.1f}°"],
    title="Analyzer axes (double-angle on Bloch x–z equator)"
)

# Rebuild the outcomes table (for plotting)
outcomes, probs = joint_outcome_table(psi, a, b)
plot_joint_probs(axs[1], outcomes, probs,
                 title=f"Born joint probs for this (a,b)\nSampled: sA={sA:+d}, sB={sB:+d}")

axs[2].axhline(0, lw=1.0)
axs[2].plot([0,1], [rA_pre[0],  rA_post[0]], marker="o", label="Alice rx")
axs[2].plot([0,1], [rA_pre[2],  rA_post[2]], marker="o", label="Alice rz")
axs[2].plot([0,1], [rB_pre[0],  rB_post[0]], marker="o", label="Bob rx")
axs[2].plot([0,1], [rB_pre[2],  rB_post[2]], marker="o", label="Bob rz")
axs[2].set_xticks([0,1]); axs[2].set_xticklabels(["pre", "post"])
axs[2].grid(True, alpha=0.3)
axs[2].set_title("Local Bloch components (x,z)\nSinglet: pre ~ 0 (maximally mixed), post conditioned")
axs[2].legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()

# ---- Figure: BEFORE (ensemble) vs AFTER (specific realized waveforms)
fig2, axs = plt.subplots(2, 2, figsize=(14, 8))

plot_mixed_state_ensemble(axs[0,0], axs[0,1], t, omega, rng=rng, n=50)

vA_alpha, vA_beta = alice_ab
vB_alpha, vB_beta = bob_ab
plot_alphabeta(axs[1,0], vA_alpha, vA_beta, title=f"After collapse: Alice αβ (sA={sA:+d})")
plot_alphabeta(axs[1,1], vB_alpha, vB_beta, title=f"After collapse: Bob αβ (sB={sB:+d})")

plt.tight_layout()
plt.show()

fig3, axs = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

vaA, vbA, vcA = alice_abc
vaB, vbB, vcB = bob_abc
plot_abc(axs[0], t, vaA, vbA, vcA, title=f"Alice abc after collapse (sA={sA:+d})")
plot_abc(axs[1], t, vaB, vbB, vcB, title=f"Bob   abc after collapse (sB={sB:+d})")

plt.tight_layout()
plt.show()

# ============================================================
# 5) Full CHSH with strict ±1 outcomes + convergence plot
# ============================================================

# Standard CHSH settings
a0, a1 = 0.0, np.pi/4
b0, b1 = np.pi/8, -np.pi/8

def estimate_E(psi, a, b, n_trials, rng):
    vals = np.empty(n_trials, dtype=float)
    for k in range(n_trials):
        r = sample_joint_outcome(psi, a, b, rng=rng)
        vals[k] = r["sA"] * r["sB"]
    return vals.mean()

def estimate_S(n_trials, seed=0):
    rng_local = np.random.default_rng(seed)
    psi = bell_singlet()
    E00 = estimate_E(psi, a0, b0, n_trials, rng_local)
    E01 = estimate_E(psi, a0, b1, n_trials, rng_local)
    E10 = estimate_E(psi, a1, b0, n_trials, rng_local)
    E11 = estimate_E(psi, a1, b1, n_trials, rng_local)
    S = abs(E00 + E01 + E10 - E11)
    return S, (E00, E01, E10, E11)

# Convergence curve
trial_grid = np.unique(np.logspace(2, 5, 18).astype(int))
S_vals = []
for n in trial_grid:
    S, _ = estimate_S(int(n), seed=42)
    S_vals.append(S)

fig4, ax = plt.subplots(figsize=(8, 4))
ax.plot(trial_grid, S_vals, marker="o", lw=1.0)
ax.axhline(2.0, ls="--", lw=1.0)
ax.axhline(2*np.sqrt(2), ls="-", lw=1.0)
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
ax.set_xlabel("Trials per setting (log scale)")
ax.set_ylabel("Estimated CHSH S")
ax.set_title("Strict ±1 CHSH with Born sampling (→ 2√2)")
plt.tight_layout()
plt.show()
